In [1]:

import os
import cv2
import numpy as np
from skimage.feature import local_binary_pattern
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Constants
train_folder = r'C:\Users\KIIT\Desktop\bone_fracture\train'
test_folder = r'C:\Users\KIIT\Desktop\bone_fracture\test'
IMG_SIZE = (128, 128)
RADIUS = 1
POINTS = 8
BIN_SIZE = 16

def load_data(directory):
    data = []
    labels = []
    label_names = sorted(os.listdir(directory))
    for label in label_names:
        class_dir = os.path.join(directory, label)
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                data.append(img)
                labels.append(label_names.index(label))
    return np.array(data), np.array(labels), label_names

# Load the dataset
train_data, train_labels, label_names = load_data(train_folder)
test_data, test_labels, _ = load_data(test_folder)

def calculate_lbp_for_channel(img_channel):
    lbp_img = local_binary_pattern(img_channel, POINTS, RADIUS, method='uniform')
    return lbp_img

def calculate_ldp_for_channel(img_channel):
    gradients = [cv2.Sobel(img_channel, cv2.CV_64F, 1, 0, ksize=3), cv2.Sobel(img_channel, cv2.CV_64F, 0, 1, ksize=3)]
    directions = np.arctan2(gradients[1], gradients[0]) * 180 / np.pi
    ldp_img = np.zeros_like(img_channel)
    for angle in [0, 45, 90, 135]:
        pattern = (directions >= angle - 22.5) & (directions < angle + 22.5)
        ldp_img += pattern.astype(np.uint8)
    return ldp_img

def extract_features(data):
    features = []
    for img in data:
        # Convert to HSV color space
        hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv_img)

        # Calculate LBP and LDP for each channel
        lbp_h = calculate_lbp_for_channel(h)
        lbp_s = calculate_lbp_for_channel(s)
        lbp_v = calculate_lbp_for_channel(v)

        ldp_h = calculate_ldp_for_channel(h)
        ldp_s = calculate_ldp_for_channel(s)
        ldp_v = calculate_ldp_for_channel(v)

        # Calculate histograms
        lbp_hist_h, _ = np.histogram(lbp_h, bins=BIN_SIZE, range=(0, BIN_SIZE))
        lbp_hist_s, _ = np.histogram(lbp_s, bins=BIN_SIZE, range=(0, BIN_SIZE))
        lbp_hist_v, _ = np.histogram(lbp_v, bins=BIN_SIZE, range=(0, BIN_SIZE))

        ldp_hist_h, _ = np.histogram(ldp_h, bins=BIN_SIZE, range=(0, BIN_SIZE))
        ldp_hist_s, _ = np.histogram(ldp_s, bins=BIN_SIZE, range=(0, BIN_SIZE))
        ldp_hist_v, _ = np.histogram(ldp_v, bins=BIN_SIZE, range=(0, BIN_SIZE))

        # Normalize histograms
        lbp_hist_h = lbp_hist_h.astype('float') / lbp_hist_h.sum()
        lbp_hist_s = lbp_hist_s.astype('float') / lbp_hist_s.sum()
        lbp_hist_v = lbp_hist_v.astype('float') / lbp_hist_v.sum()

        ldp_hist_h = ldp_hist_h.astype('float') / ldp_hist_h.sum()
        ldp_hist_s = ldp_hist_s.astype('float') / ldp_hist_s.sum()
        ldp_hist_v = ldp_hist_v.astype('float') / ldp_hist_v.sum()

        # Concatenate features
        features.append(np.concatenate((lbp_hist_h, lbp_hist_s, lbp_hist_v, ldp_hist_h, ldp_hist_s, ldp_hist_v)))

    return np.array(features)

# Extract features for train and test data
train_features = extract_features(train_data)
test_features = extract_features(test_data)

print("Train features shape:", train_features.shape)
print("Test features shape:", test_features.shape)

# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(train_features, train_labels, test_size=0.2, random_state=42)

# Initialize classifiers
classifiers = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "AdaBoost": AdaBoostClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss'),
    "CatBoost": CatBoostClassifier(verbose=0),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

# Train and evaluate each classifier
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy:.2f}")


Train features shape: (21146, 96)
Test features shape: (516, 96)
SVM Accuracy: 0.81
Random Forest Accuracy: 0.95
K-Nearest Neighbors Accuracy: 0.94
Decision Tree Accuracy: 0.91
Naive Bayes Accuracy: 0.50


C:\Users\KIIT\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy: 0.68
Gradient Boosting Accuracy: 0.91
XGBoost Accuracy: 0.95
CatBoost Accuracy: 0.94
Logistic Regression Accuracy: 0.76
